# CHADSVASC Score extraction for Jessa dataset
We use a subset of the dataset, with only AF positive entries cause those are the focus of the Quality Indicator.

In [ ]:
import re
import pandas as pd

In [ ]:
TRAIN_FILE = r"..\code\baseline\datasets\new_training_set_with_sections.feather"
VAL_FILE = r"..\code\baseline\datasets\new_val_set_with_sections.feather"
TEST_FILE = r"..\code\baseline\datasets\new_test_set_with_sections.feather"

train_df = pd.read_feather(TRAIN_FILE)
val_df = pd.read_feather(VAL_FILE)
test_df = pd.read_feather(TEST_FILE)

df = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)

print(df.shape)

In [ ]:
print("Amount of data loaded:", len(df))

In [ ]:
print(df.columns)

## Baseline: REGEX-based retrieval of literal scores

I had to at the word boundaries, because I got many false positives due to the word "schaduw".

In [ ]:
chads_pattern = r"""
    \b(?:CHA[²2?-]+?DS?[2²?-]+?|CHADVASC?|CHADS-?2?)
    [\s\-\_]*                            
    (?:VAS?C?2?)?                           
"""

chads_score_pattern = chads_pattern + r"[\s=:-]?[^\d]{0,35}?([0-9])"

In [ ]:
num_missing = df["text"].isna().sum()
num_empty = (df["text"] == "").sum()
num_blank = num_missing + num_empty

print(f"NaN values: {num_missing}")
print(f"Empty strings: {num_empty}")
print(f"Total blank entries: {num_blank}")

df = df.dropna(subset=["text"])

In [ ]:
def extract_chadsvasc(text):
    if not isinstance(text, str):
        return None
    match = re.search(chads_score_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        return int(match.group(1))

    return None

In [ ]:
df["chadsvasc_score"] = df["text"].apply(extract_chadsvasc)
print(df["chadsvasc_score"].value_counts(dropna=False))
num_non_nan = df["chadsvasc_score"].count()
print("Number of notes mentioning a CHADS-VASc score:", num_non_nan)

In [ ]:
df["chads_mentioned"] = df["text"].str.contains(
    chads_pattern, case=False, na=False, regex=True, flags=re.VERBOSE
)
num_mentions = df["chads_mentioned"].sum()
total_docs = len(df)
print(
    f"\nDocuments mentioning CHADS: {num_mentions}/{total_docs} ({num_mentions/total_docs:.2%})"
)


def get_chads_context(text, window=50):
    if not isinstance(text, str):
        return None
    matches = re.finditer(chads_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    contexts = []
    for m in matches:
        start = max(m.start() - window, 0)
        end = min(m.end() + window, len(text))
        contexts.append(text[start:end])
    return contexts if contexts else None


df["chads_context"] = df["text"].apply(get_chads_context)

missing_score_mask = df["chads_mentioned"] & (df["chadsvasc_score"].isnull())
missing_score_notes = df.loc[missing_score_mask, "text"]

print(
    f"\nNumber of notes mentioning CHADS but no extracted score: {len(missing_score_notes)}"
)

In [ ]:
window = 70

for i, note in enumerate(missing_score_notes.head(20)):
    if not isinstance(note, str):
        continue
    match = re.search(chads_pattern, note, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        start = max(match.start() - window, 0)
        end = min(match.end() + window, len(note))
        context = note[start:end]
        print(f"\nExample {i+1} (context around CHADS-VASc):\n...{context}...")
        print(f"Match: {match.group(0)}")
    else:
        print(f"\nExample {i+1} (no match found)\n")

## Qualitative data

In [ ]:
qualitative_pattern = r"""
    \b(verhoogde|hoge|lage)\b 
    [\s\-]*                                             
    """ + chads_pattern


def extract_chads_qualitative(text):
    if not isinstance(text, str):
        return None
    match = re.search(qualitative_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        adjective = match.group(1).lower()
        if adjective in ["lage"]:
            return 0
        elif adjective in ["verhoogde", "hoge"]:
            return 2
    return None


mask_no_numeric = df["chadsvasc_score"].isna()
df.loc[mask_no_numeric, "chads_qual_class"] = df.loc[mask_no_numeric, "text"].apply(
    extract_chads_qualitative
)
print("\nExtracted qualitative CHADS classes:")
print(df["chads_qual_class"].value_counts(dropna=False))

##  Greater than / > cases  

In [ ]:
threshold_pattern_greater = chads_pattern + r"""
    [^\d]{0,20}?                        
    (?:>|\bgroter\s+dan\b|\bhoger\s+dan\b|\bminstens\b)  
    \s*([0-9])                              
"""

threshold_pattern_smaller = chads_pattern + r"""
    [^\d]{0,20}?                        
    (?:<|\blager\s+dan\b|\bminder\s+dan\b)  
    \s*([0-9])                              
"""


def extract_threshold_chadsvasc(text, pattern):
    match = re.search(pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        return int(match.group(1))
    return None


### GREATER ###
df["chadsvasc_score_threshold_greater"] = df["text"].apply(
    extract_threshold_chadsvasc, args=(threshold_pattern_greater,)
)
print(df["chadsvasc_score_threshold_greater"].value_counts(dropna=False))
num_non_nan = df["chadsvasc_score_threshold_greater"].count()
print("Number of CHADS-VASc scores using a greater threshold:", num_non_nan)


df["chadsvasc_score_threshold_smaller"] = df["text"].apply(
    extract_threshold_chadsvasc, args=(threshold_pattern_smaller,)
)

In [ ]:
# find the entries that have a threshold case and overwrite the found CHADSVASC score from earlier with the old value + 1
mask_threshold = df["chadsvasc_score_threshold_greater"].notna()

# for idx in df[mask_threshold].index:
#     old = df.at[idx, 'chadsvasc_score']

#     # If old value is 2 → instead of 3, assign class 13
#     if old == 2:
#         df.at[idx, 'chadsvasc_score'] = 13
#     else:
#         # Normal increment otherwise
#         df.at[idx, 'chadsvasc_score'] = old + 1


### SMALLER ###

print(df["chadsvasc_score_threshold_smaller"].value_counts(dropna=False))
num_non_nan = df["chadsvasc_score_threshold_smaller"].count()
print("Number of CHADS-VASc scores using a smaller threshold:", num_non_nan)

# find the entries that have a threshold case and overwrite the found CHADSVASC score from earlier with the old value - 1
mask_threshold = df["chadsvasc_score_threshold_smaller"].notna()
df.loc[mask_threshold, "chadsvasc_score"] = (
    df.loc[mask_threshold, "chadsvasc_score"].fillna(0) - 1
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Prepare the three groups as before
mask_greater = df["chadsvasc_score_threshold_greater"].notna()
mask_smaller = df["chadsvasc_score_threshold_smaller"].notna()
mask_exact = df["chadsvasc_score"].notna() & ~mask_greater & ~mask_smaller

# 2. Create frequency tables for each
exact_series = df.loc[mask_exact, "chadsvasc_score"].value_counts()
greater_series = df.loc[
    mask_greater, "chadsvasc_score_threshold_greater"
].value_counts()
smaller_series = df.loc[
    mask_smaller, "chadsvasc_score_threshold_smaller"
].value_counts()

# 3. Combine into a single wide-format DataFrame for stacking
# This ensures every score level (0-9) is aligned
combined_df = (
    pd.DataFrame(
        {
            "Exact (=)": exact_series,
            "Greater than (>)": greater_series,
            "Smaller than (<)": smaller_series,
        }
    )
    .fillna(0)
    .sort_index()
)

# 4. Plotting using Pandas (with Seaborn styling)
sns.set_theme(style="whitegrid")
# Use the colors from before
colors = ["#4C72B0", "seagreen", "indianred"]

ax = combined_df.plot(
    kind="bar", stacked=True, figsize=(12, 7), color=colors, width=0.8
)

# 5. Thesis Formatting
plt.title(
    "Composition of CHA$_2$DS$_2$-VASc Mentions by Score and Style", fontsize=14, pad=20
)
plt.xlabel("Captured Numeric Value", fontsize=12)
plt.ylabel("Number of Discharge Notes", fontsize=12)
plt.xticks(rotation=0)  # Keep score numbers horizontal
plt.legend(title="Documentation Style")

# Add total count labels on top of the stacked bars
for i, (idx, row) in enumerate(combined_df.iterrows()):
    total = row.sum()
    if total > 0:
        ax.text(i, total + 2, f"{int(total)}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
print(df.columns)

In [ ]:
df = df.drop(
    [
        "chadsvasc_score_threshold_greater",
        "chadsvasc_score_threshold_smaller",
        "chads_context",
    ],
    axis=1,
)

# Component-based reconstruction


|        Component        |            Criteria / Example Keywords            | Points |         Related ICD10 codes        |
|:-----------------------:|:-------------------------------------------------:|:------:|:----------------------------------:|
| C: Congestive HF        | “heart failure”, “CHF”                            | 1      | I50*                               |
| H: Hypertension         | “hypertension”, “HTN”                             | 1      |                                    |
| A2: Age ≥75             | “age 75”, “older adult”, numeric age if available | 2      |                                    |
| D: Diabetes Mellitus    | “diabetes”, “DM”                                  | 1      |                                    |
| S2: Prior stroke/TIA/TE | “stroke”, “TIA”, “thromboembolism”, “CVA”         | 2      | I69* = Oude CVA, Z86.73 = Oude TIA |
| V: Vascular disease     | “MI”, “PAD”, “peripheral artery disease”, “CAD”   | 1      |                                    |
| A: Age 65–74            | “age 65-74”, “65 years”                           | 1      |                                    |
| Sc: Female sex          | Usually documented as “female”                    | 1      |                                    |

**Problem with this approach**  
Earlier we merged the ICD9 and ICD10 cases of AF, but we only kept the ICD10 codes.  
So only a small percentage will probably have these codes, there are false negatives who still have the old ICD9 variant. 


## Using the available ICD10 codes
Maybe we can find other “>2”/critical patients, just by using the ICD10 codes.

In [ ]:
print(df.columns)

In [ ]:
# Congestive Heart Failure columns
chf_cols = [
    "diagnose_i501",
    "diagnose_i502",
    "diagnose_i503",
    "diagnose_i504",
    "diagnose_i508",
    "diagnose_i509",
]
mask_chf = df[chf_cols].any(axis=1)

# Stroke / TIA columns
mask_stroke = (df["diagnose_i69"] == 1) | (df["diagnose_z8673"] == 1)

mask_missing_score = df["chadsvasc_score"].isna()

mask = mask_chf & mask_stroke & mask_missing_score
df_chf_stroke_missing = df[mask]

print(f"Number of notes matching criteria: {len(df_chf_stroke_missing)}")

These are patients whose ChadsVasc score would have been at least 3, if reported.   
We can easily check if there are cases where the ChadsVasc score is wrongly assigned in the discharge summary. 

In [ ]:
low_score = (
    (df["chadsvasc_score"] == 1)
    | (df["chadsvasc_score"] == 2)
    | (df["chadsvasc_score"] == 0)
)
mask = mask_chf & mask_stroke & low_score
df_chf_stroke_missing = df[mask]

print(
    f"Number of notes with a wrongly assigned ChadsVasc-score: {len(df_chf_stroke_missing)}"
)

print()

We can have a closer look at the area around the word "CHADSVAsc".

In [ ]:
def print_context(df, pattern):
    window = 250

    for i, row in enumerate(df.head(20).itertuples()):
        note = row.text
        score = row.chadsvasc_score

        if not isinstance(note, str):
            continue

        match = re.search(pattern, note, flags=re.IGNORECASE | re.VERBOSE)

        if match:
            start = max(match.start() - window, 0)
            end = min(match.end() + window, len(note))
            context = note[start:end]
            print(
                f"\nExample {i+1} ({score})\nContext around mention:\n...{context}..."
            )
        else:
            print(f"\nExample {i+1} ({score})\nNo match found\n")

In [ ]:
print_context(df_chf_stroke_missing, chads_pattern)

## Using REGEX for the other criteria

### Extracting the age from unstructured notes

In [ ]:
# age_pattern = r"""
#         \b(?:leeftijd\s*[:\-]?\s*|
#         (\d{1,3})\s*jaar\s*oud|
#         (\d{1,3})\s*-\s*jarige|
#         (\d{1,3})\s*jarige)
#     """


# def extract_age(text):
#     if not isinstance(text, str):
#         return None

#     match = re.search(age_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
#     if match:
#         for g in match.groups():
#             if g:
#                 return int(g)
#     return None


# df['age'] = df['text'].apply(extract_age)
# print(df['age'].value_counts(dropna=False))
# num_non_nan = df['age'].count()
# print("Number of non-NaN age extractions:", num_non_nan)

In [ ]:
# df_age_of_interest = df[df['age'] == 14.0]

# print_context(df_age_of_interest, age_pattern)

# df_age_of_interest = df[df['age'] == 4.0]

# print_context(df_age_of_interest, age_pattern)


# df_age_of_interest = df[df['age'] == 40.0]

# print_context(df_age_of_interest, age_pattern)

In the Dutch dataset, ages are more available. We do see the problem that unrelevant ages pop up, from the medical history etc. 
To solve this, we try to solve it by first splitting in sections.

As an additional problem, we see that a lot of age-references are made in the "Cardiale Risicofactoren" and the "Medische antecedenten" sections. We aim to exclude them in order to find the true age. Most true age-references seem to be made in the "Bespreking en besluit" section. 

### Extracting sex from unstructured notes

In [ ]:
# import re

# def extract_sex(text):
#     if not isinstance(text, str):
#         return None

#     pattern = r"""
#     \b(?:geslacht|sex)\s*[:\-]?\s*   # match 'geslacht:' of 'sex-'
#     (man|vrouw|m|v)\b                # capture male/female/M/F
#     \b(man|vrouw|m|v)\b               # standalone man/vrouw/M/V
#     """


#     match = re.search(pattern, text, flags=re.IGNORECASE | re.VERBOSE)
#     if match:
#         # Return standardized 'M' or 'F'
#         sex = next(g for g in match.groups() if g)
#         sex = sex.lower()
#         if sex.startswith('m'):
#             return 'M'
#         elif sex.startswith('f'):
#             return 'F'
#     return None

# df['sex'] = df['text'].apply(extract_sex)

# print(df['sex'].value_counts(dropna=False))

### Extracting hypertension from unstructured notes ###
Approach: first exlude the family history, cause this could infere with the result. To do this, we need to be able to split the note into segments. We can use the same strategy used in "Towards Semi-Structured Automatic ICD Coding via Tree-based Contrastive Learning", where an algorithm is developed for this purpose. It is based on document frequency-inverse average phrase frequency, with the thought that a title appear a lot of times across notes, but only once within a note.   

The code is available, so after executing, we get the following titles:

```
Top titles:
we weerhouden
obesitas
roken
glucosemetabolisme
erfelijkheid
lipiden
medische antecedenten
longen
pols
andere risicofactoren
temperatuur
zie bijlage
parameters
allergie
laboratoriumresultaten
ecg
saturatie
abdomen
```

In constrast with the mimic dataset, we don't see many meta-data related titles. 
As a next step, I manually screened some discharge notes and saw that the section titles are mainly written in capital letters. This fact could greatly reduce the extraction, since we could use a Regex to detect them. I tried that and got a list with 3385 results, since all slight variations (due to style or spelling mistakes) are counted as separate. As a final attempt to get a good result, I tried to combine both methods. By first generating the top 40 titles by the df_iapf method and seeing which one appears in the 3385 list, we are sure that the term is a frequent title. This gave the following results:

In [ ]:
# import pandas as pd
# import re
# import pprint

# section_titles = [
#     "allergie",
#     "anamnese",
#     "beleid",
#     "besluit",
#     "cardiale risicofactoren",
#     "ecg",
#     "echocardiografie",
#     "elektrolyten",
#     "klinisch onderzoek",
#     "laboratoriumresultaten",
#     "medische antecedenten",
#     "parameters",
#     "planning",
#     "rx thorax",
#     "telemetrie",
#     "thuismedicatie",
# ]

# # section_pattern = re.compile(
# #     r'(?i)^(' + '|'.join([re.escape(title) for title in section_titles]) + r')\s*[:\n]',
# #     re.MULTILINE
# # )

# section_pattern = re.compile(
#     r'^([A-Z\s]+?)\s*(?:[:\n]|$)',
#     re.MULTILINE
# )


# def split_sections(text):
#     sections = {}
#     matches = list(section_pattern.finditer(text))
#     for i, match in enumerate(matches):
#         start = match.end()
#         end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
#         section_name = match.group(1).upper()
#         sections[section_name] = text[start:end].strip()
#     return sections

# df['sections'] = df['text'].apply(split_sections)
# print(df['sections'].head())


# pprint.pprint(df['sections'].iloc[0])


# # df.to_feather(r"..\..\baseline\feather_files\jessa_notes_with_sections_column.feather")

In [ ]:
# def section_age_counts(sections):
#     counts = {}
#     for section_name, text in sections.items():
#         if not isinstance(text, str):
#             counts[section_name] = 0
#         else:
#             counts[section_name] = len(re.findall(age_pattern, text, flags=re.IGNORECASE | re.VERBOSE))
#     return counts

# df['age_counts_per_section'] = df['sections'].apply(section_age_counts)

# def section_with_most_age(counts_dict):
#     if counts_dict:
#         return max(counts_dict, key=counts_dict.get)
#     return None

# df['section_most_age'] = df['age_counts_per_section'].apply(section_with_most_age)
# print(df['section_most_age'].value_counts())

In [ ]:
# import re

# # Patterns
# chads_pattern = r"""
#     \b(?:CHA[²2?-]+?DS?[2²?-]+?|CHADVASC?|CHADS-?2?)
#     [\s\-\_]*
#     (?:VAS?C?2?)?
# """

# # Function to count CHADS-VASc mentions per section
# def section_chads_counts(sections):
#     counts = {}
#     for section_name, text in sections.items():
#         if not isinstance(text, str):
#             counts[section_name] = 0
#         else:
#             counts[section_name] = len(re.findall(chads_pattern, text, flags=re.IGNORECASE | re.VERBOSE))
#     return counts

# df['chads_counts_per_section'] = df['sections'].apply(section_chads_counts)

# # Function to get section with most mentions
# def section_most_chads(counts_dict):
#     if counts_dict:
#         max_count = max(counts_dict.values())
#         if max_count == 0:
#             return None
#         # In case multiple sections have same max, pick first
#         for sec, count in counts_dict.items():
#             if count == max_count:
#                 return sec
#     return None

# df['section_most_chads'] = df['chads_counts_per_section'].apply(section_most_chads)

# # Check the most frequent sections
# print(df['section_most_chads'].value_counts())

### Revision of age extraction

In [ ]:
relevant_sections = [
    "BESLUIT",
]


def extract_age_from_sections(sections):
    """
    sections: dict mapping section_name -> section_text
    relevant_sections: list of section names to search in (case-insensitive)
    Returns a list of all ages found in the relevant sections.
    """

    text = " ".join(str(sections.get(sec, "")).lower() for sec in relevant_sections)
    if re.search(r"\b(\d{2})-jarige\b", text):
        age = int(re.search(r"\b(\d{2})-jarige\b", text).group(1))
        return age
    return None


df["age"] = df["sections"].apply(extract_age_from_sections)
print(df["age"].value_counts(dropna=False))
num_non_nan = df["age"].count()

In [ ]:
target_ages = [58, 3]


def get_besluit_section(sections):
    if not isinstance(sections, dict):
        return None
    return sections.get("BESPREKING EN BESLUIT", None)


df_target = df[df["age"].apply(lambda ages: sorted(ages) == sorted(target_ages))]

for idx, row in df_target.iterrows():
    first_sections = df_target["sections"].iloc[0]

    print(first_sections.keys())
    print(df_target["sections"].keys())
    print(f"--- Row {idx} ---")
    print(get_besluit_section(row["sections"]))
    print()

Temporary solution: select the first age that pops up.

In [ ]:
df["age"] = df["age"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

We redid the query for age, and the same should probably happen for sex as well (since family history can again infer with a correct extraction). This is still a todo. From some initial exploration, we find that maybe using the gender of the word "patient" is a better indicator then relying on a regex. 

In [ ]:
def print_first_tokens(section_text, n_tokens=60):
    if not isinstance(section_text, str):
        return
    tokens = section_text.split()
    snippet = " ".join(tokens[:n_tokens])
    print(snippet)


for idx, row in df.head(5).iterrows():
    besluit_text = row["sections"].get("BESPREKING EN BESLUIT")
    if besluit_text:
        print(f"--- Row {idx} ---")
        print_first_tokens(besluit_text, n_tokens=60)

### Hypertension

In [ ]:
relevant_sections = ["BESLUIT", "BESPREKING EN BESLUIT"]

In [ ]:
import re

HYPERTENSION_PATTERNS = [
    r"\bhypertensie\b",
    r"\bhoge bloeddruk\b",
    r"\bverhoogde bloeddruk\b",
]

NEGATION_PATTERNS = [
    r"\bgeen hypertensie\b",
    r"\blage bloeddruk\b",
]

In [ ]:
def detect_patterns(sections, positive_patterns, negative_patterns):

    text = " ".join(
        sections[sec].lower()
        for sec in relevant_sections
        if sections.get(sec) is not None
    )
    if not text.strip():
        return None

    if any(re.search(p, text) for p in negative_patterns):
        return 0
    if any(re.search(p, text) for p in positive_patterns):
        return 1
    return None

In [ ]:
df["hypertension"] = df["sections"].apply(
    detect_patterns, args=(HYPERTENSION_PATTERNS, NEGATION_PATTERNS)
)

print(df["hypertension"].value_counts(dropna=False))
num_non_nan = df["hypertension"].count()
print("Number of non-NaN hypertension scores:", num_non_nan)

### Diabetes extraction

In [ ]:
relevant_sections = ["RISICOFACTOREN", "BESLUIT", "BESPREKING EN BESLUIT"]

In [ ]:
# Common diabetes mentions
DIABETES_PATTERNS = [
    # r'\bdiabetes\b',
    # r'\bdiabetes mellitus\b',
    # r'\btype\s*[12]\s*diabetes\b',
    # r'\btype\s*[12]\s*dm\b',
    # r'\bdm\b',  # abbreviation for diabetes mellitus
    # r'\bnon[-\s]*insulin[-\s]*dependent\b',
    # r'\binsulin[-\s]*dependent\b',
    # r'\bdiabetisch\b',  # e.g., "diabetic neuropathy"
    r"glucosemetabolisme\b",
]

# Negations / exclusions
DIABETES_NEGATION_PATTERNS = [
    r"\bgeen diabetes\b",
    r"\bnegative for diabetes\b",
    r"\bgeen geschiedenis van diabetes\b",
    r"\bniet[-\s]*diabetisch\b",
]

In [ ]:
print(df.columns)

In [ ]:
df["diabetes"] = df["sections"].apply(
    detect_patterns, args=(DIABETES_PATTERNS, DIABETES_NEGATION_PATTERNS)
)
print(df["diabetes"].value_counts(dropna=False))
num_non_nan = df["diabetes"].count()
print("Number of non-NaN diabetes scores:", num_non_nan)

In [ ]:
import re


def get_context(sections, keyword="glucosemetabolisme", window=4):
    # Combine the relevant text (using the fix from before)
    text = " ".join(str(sections[s]) for s in relevant_sections if sections.get(s))

    # Search for the keyword
    match = re.search(re.escape(keyword), text, re.IGNORECASE)

    if match:
        start = max(0, match.start())
        end = min(len(text), match.end() + window)
        return f"...{text[start:end]}...\n"

    return "Keyword not found in text"


# 1. Filter the DataFrame for the '1' cases
positive_cases = df[df["diabetes"] == 1].copy()

# 2. Apply the context extractor
positive_cases["snippet"] = positive_cases["sections"].apply(
    get_context, keyword="glucosemetabolisme", window=60
)

# 3. Print the results
for index, row in positive_cases.iterrows():
    print(f"Row {index}: {row['snippet']}\n")

### Extract cardivascular disease

In [ ]:
# Common vascular disease mentions
VASCULAR_DISEASE_PATTERNS = [
    r"\bvascular disease\b",
    r"\bperipheral (artery|arterial|vascular) disease\b",
    r"\bpad\b",  # abbreviation for peripheral artery disease
    r"\bperipheral atherosclerosis\b",
    r"\bcoronary artery disease\b",
    r"\bcad\b",
    r"\bischemic heart disease\b",
    r"\bmyocardial infarction\b",
    r"\bmi\b",
    r"\baortic plaque\b",
    r"\batherosclerosis\b",
    r"\bangioplasty\b",
    r"\bbypass surgery\b",
    r"\bcarotid stenosis\b",
]

# Negations and exclusions
VASCULAR_DISEASE_NEGATION_PATTERNS = [
    r"\bno vascular disease\b",
    r"\bdenies vascular disease\b",
    r"\bwithout vascular disease\b",
    r"\bno evidence of vascular disease\b",
    r"\bno (history|hx) of vascular disease\b",
    r"\bno cad\b",
    r"\bdenies cad\b",
    r"\bno mi\b",
    r"\bno history of (mi|myocardial infarction)\b",
    r"\bno (pad|peripheral artery disease)\b",
    r"\bnon[-\s]*ischemic\b",
]

In [ ]:
df["cardivascular"] = df["sections"].apply(
    detect_patterns,
    args=(VASCULAR_DISEASE_PATTERNS, VASCULAR_DISEASE_NEGATION_PATTERNS),
)
print(df["cardivascular"].value_counts(dropna=False))
num_non_nan = df["cardivascular"].count()
print("Number of non-NaN cardivascular scores:", num_non_nan)

## Inferred CHADSVASC score

In [ ]:
chf_cols = [
    "diagnose_i501",
    "diagnose_i502",
    "diagnose_i503",
    "diagnose_i504",
    "diagnose_i508",
    "diagnose_i509",
]
mask_chf = df[chf_cols].any(axis=1).astype(int)

mask_stroke = ((df["diagnose_i69"] == 1) | (df["diagnose_z8673"] == 1)).astype(int)

sex = (df["sex"].fillna("M").str.upper() == "F").astype(int)

age_1 = ((df["age"] >= 65) & (df["age"] <= 74)).astype(int)
age_2 = (df["age"] >= 75).astype(int)

# Compute CHA2DS2-VASc
df["inferred_chadsvasc"] = (
    mask_chf
    + df["hypertension"].fillna(0).astype(int)
    + df["diabetes"].fillna(0).astype(int)
    + mask_stroke * 2
    + df["cardivascular"].fillna(0).astype(int)
    + sex
    + age_1
    + 2 * age_2
)

print(df["inferred_chadsvasc"].value_counts(dropna=False))

In [ ]:
df["chadsvasc_score"] = pd.to_numeric(df["chadsvasc_score"], errors="coerce")

df_compare = df.dropna(subset=["chadsvasc_score"]).copy()

df_compare["score_diff"] = (
    df_compare["inferred_chadsvasc"] - df_compare["chadsvasc_score"]
)

print(df_compare[["inferred_chadsvasc", "chadsvasc_score", "score_diff"]].describe())

print("Exact match rate:", (df_compare["score_diff"] == 0).mean())
print("Difference counts:")
print(df_compare["score_diff"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure the DataFrame is clean
df_compare = df.dropna(subset=["chadsvasc_score"]).copy()
df_compare["score_diff"] = (
    df_compare["inferred_chadsvasc"] - df_compare["chadsvasc_score"]
)

# 1. Distribution of score differences
plt.figure(figsize=(8, 5))
sns.countplot(x="score_diff", data=df_compare, color="skyblue")
plt.title("Distribution of Difference: Inferred - Documented CHADS-VASc")
plt.xlabel("Score Difference")
plt.ylabel("Count")
plt.show()

# Comparison with ground truth labels

In [ ]:
print(df["label"].value_counts())

In [ ]:
print(df.columns)

matches = df["label"] == df["chadsvasc_score"]
num_matches = matches.sum()
total = len(df)
accuracy = num_matches / total

print(f"Number of matches: {num_matches}")
print(f"Total rows: {total}")
print(f"Agreement rate: {accuracy:.2%}")

In [ ]:
conf_matrix = pd.crosstab(df["label"], df["chadsvasc_score"])
print(conf_matrix)

### Analyse: the limits of regexes
The regex has problems capturing slight variations, like spelling mistakes. 

In [ ]:
non_matches = df[df["label"] != df["chadsvasc_score"]]
print(f"Number of non-matching rows: {len(non_matches)} / {len(df)}")
print(non_matches[["label", "chadsvasc_score", "text"]].head(10))

In [ ]:
import re
import textwrap

window = 75
line_width = 100

non_matches = df[df["label"] != df["chadsvasc_score"]]

for i, row in enumerate(non_matches.head(5).itertuples()):
    text = row.text
    label = row.label
    predicted = row.chadsvasc_score

    print(f"\n=== Non-Match {i+1} ===")
    print(f"Label (gold): {label}")
    print(f"Extracted CHADS-VASc: {predicted}\n")

    match = re.search(chads_pattern, text, flags=re.IGNORECASE | re.VERBOSE)
    if match:
        start = max(match.start() - window, 0)
        end = min(match.end() + window, len(text))
        context = text[start:end]
        wrapped_context = "\n".join(textwrap.wrap(context, width=line_width))
        print(f"Context around CHADS-VASc:\n{wrapped_context}")
        print(f"Matched phrase: {match.group(0)}")
    else:
        wrapped_text = "\n".join(textwrap.wrap(text, width=line_width))
        print(f"No CHADS-VASc match found. Note text:\n{wrapped_text}")